In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
# Load data with explicit types
df = pd.read_csv(
    'sales_data.csv',
    parse_dates=['date'],
    dtype={
        'product_id': 'int32',
        'region': 'category',
        'sales_amount': 'float64'
    }
)

print(f"Dataset shape: {df.shape}")
print()
print("Data types:")
print(df.dtypes)
print()
print("First 5 rows:")
print(df.head())


In [ ]:
# Calculate descriptive statistics for sales by region
region_stats = (
    df.groupby('region')['sales_amount']
    .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    .round(2)
    .sort_values('mean', ascending=False)
)

print("Descriptive Statistics for Sales by Region:")
print("=" * 50)
print(region_stats)


In [ ]:
# Identify sales records more than 2 standard deviations from the mean
overall_mean = df['sales_amount'].mean()
overall_std = df['sales_amount'].std()

df['z_score'] = (df['sales_amount'] - overall_mean) / overall_std
outliers = df[np.abs(df['z_score']) > 2]

print(f"Overall mean: ${overall_mean:.2f}")
print(f"Overall std:  ${overall_std:.2f}")
print(f"Threshold (mean ± 2*std): ${overall_mean - 2*overall_std:.2f} to ${overall_mean + 2*overall_std:.2f}")
print()
print(f"Number of outliers (>2 std from mean): {len(outliers)}")
print(f"Percentage of outliers: {100 * len(outliers) / len(df):.2f}%")
print()
print("Outlier records:")
print(outliers[['date', 'product_id', 'region', 'sales_amount', 'z_score']].round(2).head(10))


In [ ]:
# Create high-quality visualization using Matplotlib OO API
total_sales_by_region = df.groupby('region')['sales_amount'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bars = ax.bar(
    total_sales_by_region.index,
    total_sales_by_region.values,
    color=colors,
    edgecolor='black',
    linewidth=0.5
)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.annotate(
        f'${height:,.0f}',
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha='center', va='bottom',
        fontsize=11,
        fontweight='bold'
    )

ax.set_title('Total Sales by Region (2025)', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Region', fontsize=12)
ax.set_ylabel('Total Sales Amount ($)', fontsize=12)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_axisbelow(True)  # Gridlines behind bars

# Format y-axis as currency
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('sales_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTotal Sales by Region:")
for region, total in total_sales_by_region.items():
    print(f"  {region}: ${total:,.2f}")
